In [ ]:
"""
Phase 5: Topic modeling (BERTopic) on the 38K reviews + merge with Phase 4's
sentiment output into the final aspect_sentiment_table that MCP tools query directly.
 
Run in a Kaggle notebook (GPU recommended, needs internet to download the
embedding model).
 
Requires (Kaggle): pip install bertopic sentence-transformers pandas tqdm
"""

In [1]:
pip install bertopic sentence-transformers pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 1.8 MB/s eta 0:00:0000:010:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [3]:
# ---------------- CONFIG (edit to match your files) ----------------
# NOTE: paths assume the "car-review-chatbot" Kaggle Dataset is organized
# into raw/ and interim/ subfolders mirroring this repo's data/raw and
# data/interim -- re-upload/organize the Kaggle dataset to match before running.
MAIN_PATH = "/kaggle/input/datasets/raghavenderg/car-review-chatbot/interim/38000_processed.csv"
SENTIMENT_PATH = "/kaggle/input/datasets/raghavenderg/car-review-chatbot/interim/phase4_sentiment_predictions_38k.csv"
REVIEW_COL = "review"
ID_COL = "review_id"
BRAND_COL = "brand_name"
YEAR_COL = "car_year"
RATING_COL = "Rating"
MIN_WORD_COUNT = 3        # reviews at or below this word count are skipped - too noisy for topics
TARGET_TOPIC_COUNT = 40   # reduce to roughly this many topics so labels stay interpretable
# ---------------------------------------------------------------------

In [4]:
df = pd.read_csv(MAIN_PATH)
if ID_COL not in df.columns:
    df[ID_COL] = df.index
 
df["word_count"] = df[REVIEW_COL].fillna("").apply(lambda x: len(str(x).split()))
topic_input = df[df["word_count"] > MIN_WORD_COUNT].copy()
skipped = df[df["word_count"] <= MIN_WORD_COUNT].copy()
print(f"Reviews used for topic modeling: {len(topic_input)}")
print(f"Reviews skipped (too short/missing): {len(skipped)}")
 
docs = topic_input[REVIEW_COL].fillna("").astype(str).tolist()

Reviews used for topic modeling: 31769
Reviews skipped (too short/missing): 169


In [5]:
# ============ FIT BERTOPIC ============
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model, min_topic_size=30, calculate_probabilities=False)
 
topics, _ = topic_model.fit_transform(docs)

# Reduce to a manageable, interpretable number of topics
current_n_topics = len(topic_model.get_topic_info()) - 1  # exclude the -1 outlier row
if current_n_topics > TARGET_TOPIC_COUNT:
    print(f"Reducing from {current_n_topics} to ~{TARGET_TOPIC_COUNT} topics for interpretability")
    topic_model.reduce_topics(docs, nr_topics=TARGET_TOPIC_COUNT)
    topics = topic_model.topics_
 
# Reassign as many outlier (-1) documents as possible to a real topic, then refresh
# the keyword representations so get_topic_info()/get_topic() reflect the reassignment
new_topics = topic_model.reduce_outliers(docs, topics)
topic_model.update_topics(docs, topics=new_topics)
topic_input["topic_id"] = new_topics
 
n_outliers = sum(1 for t in new_topics if t == -1)
print(f"Remaining unclassified (-1) after outlier reduction: {n_outliers} "
      f"({n_outliers / len(new_topics) * 100:.1f}%)")

# ============ SAVE MODEL ============
topic_model.save("phase5_bertopic_model")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Reducing from 102 to ~40 topics for interpretability


2026-09-12 09:45:44,959 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-09-12 09:45:46,609 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Remaining unclassified (-1) after outlier reduction: 0 (0.0%)


In [8]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

MAIN_PATH = "/kaggle/input/datasets/raghavenderg/car-review-chatbot/interim/38000_processed.csv"
REVIEW_COL = "review"
MIN_WORD_COUNT = 3

df = pd.read_csv(MAIN_PATH)
df["word_count"] = df[REVIEW_COL].fillna("").apply(lambda x: len(str(x).split()))
docs = df[df["word_count"] > MIN_WORD_COUNT][REVIEW_COL].fillna("").astype(str).tolist()

topic_model = BERTopic.load("/kaggle/input/models/raghavenderg/topic-modelling/transformers/default/1/phase5_bertopic_model")

# strip stopwords + brand names (redundant with brand_name column) + allow 2-3 word phrases
brand_stopwords = [b.lower() for b in df["brand_name"].dropna().unique()]
custom_stopwords = list(ENGLISH_STOP_WORDS) + brand_stopwords + ["car", "vehicle"]
vectorizer_model = CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 3), min_df=10)

topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

topic_info = topic_model.get_topic_info()
topic_info["top_keywords"] = topic_info["Topic"].apply(
    lambda t: ", ".join([w for w, _ in topic_model.get_topic(t)][:8]) if t != -1 else "outlier/unclassified"
)
topic_info = topic_info[["Topic", "Count", "top_keywords"]].rename(
    columns={"Topic": "topic_id", "Count": "review_count"}
)
topic_info.to_csv("/kaggle/working/phase5_topic_info.csv", index=False)
print(topic_info.to_string(index=False))

 topic_id  review_count                                                                                                top_keywords
        0          3623                                                              suv, rx, great, like, ride, luxury, love, best
        1          4246                                                          drive, like, driving, m3, just, great, series, new
        2          2794                                                             a4, a6, quattro, s4, drive, great, q5, interior
        3          3689                                                     fun, best, great, drive, fun drive, love, owned, luxury
        4          2595                                               miles, 000, 000 miles, problems, replaced, bought, years, new
        5          1782                                              mercedes, benz, mercedes benz, miles, german, new, cars, years
        6          1206                                                     

In [9]:
# ============ MANUAL STEP: fill this in after inspecting phase5_topic_info.csv ============
TOPIC_LABEL_MAP = {
    0: "lexus rx & luxury suv experience",
    1: "bmw 3 series & sedan driving dynamics",
    2: "audi sedan & quattro awd performance",
    3: "general driving enjoyment & luxury ownership",
    4: "high mileage & maintenance reliability",
    5: "mercedes-benz brand ownership",
    6: "midsize luxury suvs (bmw x5/x3 & infiniti fx)",
    7: "mercedes roadsters & coupes (slk/clk/sl/amg)",
    8: "mercedes e-class & executive sedans",
    9: "mercedes ml-class suvs",
    10: "infiniti g37 & luxury utility crossovers",
    11: "nissan dealer service & purchase experience",
    12: "infotainment, navigation & audio systems",
    13: "engine oil, warranty & dealer maintenance",
    14: "compact roadsters (bmw z3/z4 & porsche boxster)",
    15: "infiniti g35 coupe & sport performance",
    16: "infiniti qx series & full-size suvs",
    17: "fuel economy & diesel mileage",
    18: "winter driving, snow & awd traction",
    19: "sports sedans & high-performance models (m5/m45)",
    20: "legacy entry-luxury sedans (infiniti i30 & lexus is300)",
    21: "audi tt roadster & sport performance",
    22: "wind noise, vibration & cabin acoustics",
    23: "seating comfort, legroom & cargo trunk space",
    24: "ride comfort & suspension smoothness",
    25: "handling quality & driving dynamics ('drives like a dream')",
    26: "interior rattles, squeaks & cabin fit-and-finish",
    27: "tires & run-flat tire performance",
    28: "sports roadsters & convertible driving fun",
    29: "bmw brand perception ('ultimate driving machine')",
    30: "transmission, shifting & gear responsiveness",
    31: "audi allroad & luxury wagons",
    32: "bmw enthusiast ownership ('bimmer' sentiment)",
    33: "lemon law, major defects & repeated repair visits",
    34: "cup holder ergonomics & interior storage",
    35: "brake performance, noise & pad wear",
    36: "acceleration, power & top speed performance",
    37: "electric vehicle range & charging infrastructure",
    38: "exterior styling, color & aesthetic appeal"
}
 
topic_info["topic_label"] = topic_info["topic_id"].map(TOPIC_LABEL_MAP)
topic_info["topic_label"] = topic_info["topic_label"].fillna("Unlabeled - review phase5_topic_info.csv")
topic_info.loc[topic_info["topic_id"] == -1, "topic_label"] = "No clear topic"

In [12]:
# ============ BUILD FINAL aspect_sentiment_table ============
 
topic_assignments = topic_input[[ID_COL, "topic_id"]]
sentiment = pd.read_csv(SENTIMENT_PATH)
 
metadata_cols = [c for c in [ID_COL, BRAND_COL, YEAR_COL, RATING_COL] if c in df.columns]
metadata = df[metadata_cols]
 
aspect_sentiment_table = (
    metadata
    .merge(sentiment, on=ID_COL, how="left")
    .merge(topic_assignments, on=ID_COL, how="left")
    .merge(topic_info[["topic_id", "top_keywords", "topic_label"]], on="topic_id", how="left")
)
 
# Reviews skipped from topic modeling (too short) get an explicit label, not a silent NaN
aspect_sentiment_table.loc[
    aspect_sentiment_table[ID_COL].isin(skipped[ID_COL]), "topic_label"
] = "Not analyzed (review too short)"
 
aspect_sentiment_table.to_csv("/kaggle/working/aspect_sentiment_table.csv", index=False)
print(f"\nDone. aspect_sentiment_table.csv saved with {len(aspect_sentiment_table)} rows.")


Done. aspect_sentiment_table.csv saved with 31938 rows.


In [11]:
# ============ PROXY CHECK: predicted sentiment vs star rating ============
# The reference set had no positive examples (see Phase 4), so this is the only
# available sanity check for the model's positive-class behavior.

if RATING_COL in aspect_sentiment_table.columns:
    print("\n--- Predicted sentiment vs average star rating (sanity check) ---")
    print(aspect_sentiment_table.groupby("predicted_sentiment")[RATING_COL].mean().round(2))
    print("Expectation: 'positive' should have the highest average rating, "
          "'negative' the lowest. If not, treat model output with caution.")


--- Predicted sentiment vs average star rating (sanity check) ---
predicted_sentiment
negative    3.32
neutral     4.33
positive    4.73
Name: Rating, dtype: float64
Expectation: 'positive' should have the highest average rating, 'negative' the lowest. If not, treat model output with caution.
